In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error




import warnings
warnings.filterwarnings('ignore')


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

data_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(data_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
# Delivery Time distribution
condition_counts = df['Delivery_Time'].value_counts()
plt.figure(figsize=(10, 5))
plt.bar(condition_counts.index, condition_counts.values, color='coral')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Task 1: Write your code here:

df_clean = df.copy()
df_clean = df_clean.drop(columns=['Order_ID'])
df_clean

In [ ]:
# Task 2: Write your code here:
missing_values = df_clean.isnull().sum()
print(missing_values)


In [ ]:
# Task 2: Continued
# i will fill the Weather, Traffic_Level, Time_of_Day by mode
df_clean['Weather'] = df_clean['Weather'].fillna(df_clean['Weather'].mode()[0])
df_clean['Traffic_Level'] = df_clean['Traffic_Level'].fillna(df_clean['Traffic_Level'].mode()[0])
df_clean['Time_of_Day'] = df_clean['Time_of_Day'].fillna(df_clean['Time_of_Day'].mode()[0])

# i will fill the Courier_Experience_yrs, Delivery_Time by mean
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mean())
df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].mean())


In [ ]:
# Task 2: Continued
missing_values = df_clean.isnull().sum()
print(missing_values)

In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df_clean):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 5: Write your code here:

numerical_cols =  df_clean.select_dtypes(include=["number"]).columns.drop("Delivery_Time")

scaler = StandardScaler()


df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])

df_clean.head()


In [ ]:
# Task 6: Write your code here:

df['Delivery_Time'].hist()

#there is allitle bit of imballance

In [ ]:
# Task 1: Write your code here:
target_column = "Delivery_Time"

X = df_clean.drop(target_column, axis=1)
y = df_clean[target_column]

In [ ]:
from sklearn.linear_model import LinearRegression
# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define Model
model = RandomForestRegressor()

# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mse_scores = []
mae_scores = []

for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics

    mae_scores.append(mean_absolute_error(y_test, y_pred))


# Print Evaluation Metrics
print("\nModel Evaluation Metrics (K-Fold)\n" + "-"*40)

print(f"MAE : {np.mean(mae_scores):.2f}")

print("-"*40)

In [ ]:
df_clean.info()

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level',
                'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Plot Predictions vs Ground Truth
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    linewidth=2
)

plt.xlabel("Actual Delivery Time (Ground Truth)")
plt.ylabel("Predicted Delivery Time")
plt.title("Linear Regression: Predictions vs Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Task Bonus: Write your code here: